# 颜色抓取玩法  Color Grab play

### 导入头文件 Import head file

In [1]:
#!/usr/bin/env python
# coding: utf-8
import cv2 as cv
import threading
import ipywidgets as widgets
from IPython.display import display
from color_grab import Color_Grab
from dofbot_utils.fps import FPS
from dofbot_utils.robot_controller import Robot_Controller

### 初始化机械臂位置 Initialize DOFBOT position

In [2]:
grab = Color_Grab()
robot = Robot_Controller()
robot.move_look_front()

In [3]:
fps = FPS()
model = 'Start'

### 创建实例,初始化参数 Create the instance and initialize the parameters

### 创建控件 Creating widget

In [4]:
# 创建控件布局  Create widget layout
button_layout  = widgets.Layout(width='200px', height='70px', align_self='center')
# 输出打印  Output printing
output = widgets.Output()
# 退出按钮  exit button
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
# 图像控件  Image widget
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='center'))
# 垂直放置  Vertical placement
controls_box = widgets.VBox([imgbox, exit_button], layout=widgets.Layout(align_self='center'))
# ['auto', 'flex-start', 'flex-end', 'center', 'baseline', 'stretch', 'inherit', 'initial', 'unset']

### 控制按钮  Control button

In [5]:
def exit_button_Callback(value):
    global model
    model = 'Exit'
#     with output: print(model)
exit_button.on_click(exit_button_Callback)

### 主程序 Main process

In [6]:
def camera():
    # 打开摄像头 Open camera
    capture = cv.VideoCapture(0, cv.CAP_V4L2)
    capture.set(cv.CAP_PROP_FRAME_WIDTH, 640)
    capture.set(cv.CAP_PROP_FRAME_HEIGHT, 480)
    while capture.isOpened():
        try:
            if model == 'Exit':
                capture.release()
                break
            _, img = capture.read()
            fps.update_fps()
            # 获得运动信息  Get motion information
            img = grab.start_grab(img)
            fps.show_fps(img)
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except KeyboardInterrupt:capture.release()
        except Exception as e:
            print(e)

### 启动  Start

In [7]:
display(controls_box,output)
threading.Thread(target=camera, ).start()

Output()